# Teste Técnico – Engenheiro de Dados

**Autor:** Carlos Silvestre

Resolução dos 4 desafios de SQL propostos no desafio técnico. 


In [0]:
%pip install pandasql "sqlalchemy<2.0"

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.6 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1.6/1.6 MB 64.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.5 MB/s eta 0:00:00
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26772 sha256=e8d570b8aaf6783c4f894a4c17d5881d9c25a639bc6fed7ef109cc0900753524
  Stored in directory: /home/spark-3cad9a6d-5df2-47a1-94f0-e9/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.52
    Not uninstalling sqlalchemy at /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3cad9a6d-5df2-47a1-94f0-e9abb8a7a9c1
    Can't uninstall 'SQLAlchemy'. No files were f

In [0]:
%restart_python

### Nota

Este notebook foi originalmente criado no **Databricks** (`%pip install` e `%restart_python` são comandos do Databricks). Para rodar no **Google Colab**:

- Remova ou ignore a célula `%restart_python`.
- Faça upload dos 6 arquivos `.csv` para `/content/` (ou ajuste os caminhos `pd.read_csv(...)` para apontar para onde os arquivos estiverem).

Veja o `README.md` do repositório para instruções passo a passo.


In [0]:
import pandas as pd
from pandasql import sqldf

# Carrega os CSVs
buyers = pd.read_csv('buyers.csv', sep=',')
order_items = pd.read_csv('order_items.csv', sep=',')
orders = pd.read_csv('orders.csv', sep=',')
payments = pd.read_csv('payments.csv', sep=',')
products = pd.read_csv('products.csv', sep=',')
sellers = pd.read_csv('sellers.csv', sep=',')

# Cria o pysqldf
pysqldf = lambda q: sqldf(q, globals())

#### Desafio 1 (Faturamento mensal)

In [0]:
# Desafio 1
# Faturamento bruto mensal dos últimos 12 meses, considerando apenas pedidos
# 'completed' ou 'delivered' (exclui 'cancelled', 'refunded' e 'processing').
query = """
    WITH faturamento_mensal AS (
        SELECT
            strftime('%Y-%m', created_at) AS mes,
            ROUND(SUM(total_value), 2) AS faturamento_bruto,
            COUNT(DISTINCT id) AS qtd_pedidos
        FROM
            orders
        WHERE
            status IN ('completed', 'delivered')
        GROUP BY
            mes
    )

    SELECT
        mes,
        faturamento_bruto,
        qtd_pedidos,
        ROUND(faturamento_bruto / NULLIF(qtd_pedidos, 0), 2) AS ticket_medio
    FROM
        faturamento_mensal
    ORDER BY
        mes DESC
    LIMIT 
        12
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1

,mes,faturamento_bruto,qtd_pedidos,ticket_medio
0,2024-11,56710238.63,3643,15566.91
1,2024-10,62493439.13,3863,16177.44
2,2024-09,60274462.50,3779,15949.84
3,2024-08,60102539.82,3743,16057.32
4,2024-07,59769406.95,3862,15476.28
5,2024-06,58797774.37,3644,16135.50
6,2024-05,60730315.72,3848,15782.31
7,2024-04,57779312.59,3653,15816.95
8,2024-03,61525913.64,3877,15869.46
9,2024-02,57926376.26,3628,15966.48


#### Desafio 2 (Crescimento de GMV por seller) 

In [0]:
# Desafio 2
# Top 10 sellers com maior crescimento de GMV entre o trimestre atual e o anterior,
# restrito a sellers com >= 50 pedidos em cada um dos dois trimestres.

query = """
    WITH periodo AS (
        SELECT
            CONCAT(STRFTIME('%Y', MAX(created_at)), '-Q', ((CAST(STRFTIME('%m', MAX(created_at)) AS INTEGER) - 1) / 3 + 1) )AS trimestre_atual,
            CONCAT(STRFTIME('%Y', DATE(MAX(created_at), '-3 months')),'-Q', ((CAST(STRFTIME('%m', DATE(MAX(created_at), '-3 months')) AS INTEGER) - 1) / 3 + 1)) AS trimestre_anterior
        FROM
            orders
    ),

    faturamento_trimestral AS (
        SELECT
            seller_id,
            CONCAT(STRFTIME('%Y', created_at), '-Q', ((CAST(STRFTIME('%m', created_at) AS INTEGER) - 1) / 3 + 1)) AS trimestre,
            ROUND(SUM(total_value), 2) AS faturamento_bruto,
            COUNT(DISTINCT id) AS qtd_pedidos,
            ROUND(SUM(total_value) / COUNT(DISTINCT id), 2) AS ticket_medio
        FROM
            orders
        WHERE
            status <> 'cancelled'
            AND STRFTIME('%Y', created_at) = (SELECT MAX(STRFTIME('%Y', created_at)) FROM orders)
        GROUP BY
            seller_id, trimestre
    ),

    seller_trimestre AS (
        SELECT 
            s.id,
            s.name AS nome,
            s.state AS estado,
            SUM(ft.faturamento_bruto) FILTER  (WHERE ft.trimestre = p.trimestre_anterior) AS gmv_trimestre_anterior,
            SUM(ft.faturamento_bruto) FILTER (WHERE ft.trimestre = p.trimestre_atual) AS gmv_trimestre_atual,
            SUM(ft.qtd_pedidos) FILTER  (WHERE ft.trimestre = p.trimestre_anterior) AS qtd_pedidos_trimestre_anterior,
            SUM(ft.qtd_pedidos) FILTER (WHERE ft.trimestre = p.trimestre_atual) AS  qtd_pedidos_trimestre_atual
        FROM 
            sellers AS s
            LEFT JOIN faturamento_trimestral AS ft ON s.id = ft.seller_id
            CROSS JOIN periodo AS p
        GROUP BY 
            s.id, s.name, s.state
    )

    SELECT 
        nome,
        estado,
        gmv_trimestre_anterior,
        gmv_trimestre_atual,
        ROUND(((gmv_trimestre_atual - gmv_trimestre_anterior) / NULLIF(gmv_trimestre_anterior, 0)) * 100, 2) AS percentual_crescimento
    FROM 
        seller_trimestre
    WHERE 
        qtd_pedidos_trimestre_anterior >= 50 AND qtd_pedidos_trimestre_atual >= 50
    ORDER BY
        percentual_crescimento DESC
    LIMIT 10
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2


# Problema identificado: O pedido mais recente da base é de 29/11/2024. Dessa forma, o "trimestre atual" (Q4/2024) contém apenas cerca de 2 meses de dados (outubro e novembro), enquanto o "trimestre anterior" (Q3/2024) está completo, com 3 meses de dados. Comparar um período parcial com um período completo faz com que o GMV do trimestre atual pareça artificialmente menor.

# Resolução: A comparação entre "trimestre atual vs. trimestre anterior" é uma métrica de calendário mais familiar para quem consulta o relatório e mais fácil de explicar, por exemplo: "Q4 vs. Q3".
# Por outro lado, janelas móveis de 90 dias podem proporcionar uma comparação mais equilibrada quando o período mais recente está incompleto, mas exigem deixar claro no relatório que não representam exatamente um "trimestre civil".
# Nesse caso, vale alinhar com o time financeiro qual das duas abordagens faz mais sentido para a análise do negócio.

,nome,estado,gmv_trimestre_anterior,gmv_trimestre_atual,percentual_crescimento
0,Costa & Ferreira Atacado Distribuidora,PE,820853.26,862340.40,5.05
1,Rodrigues & Oliveira Atacado Distribuidora,RS,983094.06,950623.42,-3.30
2,Almeida & Almeida Alimentos Distribuidora,RS,1131649.63,1049371.65,-7.27
3,Rodrigues & Almeida Atacado Distribuidora,MG,1061092.94,980769.51,-7.57
4,Costa & Santos Alimentos Distribuidora,MG,1068621.24,982430.50,-8.07
5,Santos & Silva Mercado Distribuidora,MT,1133239.60,1038129.53,-8.39
6,Santos & Silva Suprimentos Distribuidora,MS,1137631.23,1031414.42,-9.34
7,Nascimento & Souza Alimentos Distribuidora,GO,1086642.68,984738.00,-9.38
8,Costa & Silva Alimentos Distribuidora,BA,1259930.10,1132414.00,-10.12
9,Souza & Rodrigues Grupo Distribuidora,PB,1130246.57,1011511.36,-10.51


#### Desafio 3 (Descontos abusivos)****

In [0]:
# Desafio 3
# Pedidos em que o desconto total dos itens representa mais de 40% do valor
# bruto do pedido (possível indício de desconto abusivo para inflar volume).
query = """
WITH montante AS (
    SELECT 
        ord.id,
        ord.seller_id,
        ord.created_at AS pedido_data,
        ROUND(SUM(it.qty * it.unit_price), 2) AS pedido_total,
        ROUND(SUM(it.discount), 2) AS desconto_total,
        ROUND(SUM(it.discount) / NULLIF(SUM(it.qty * it.unit_price), 0) * 100, 2) AS percentual_desconto 
    FROM 
        orders AS ord
        LEFT JOIN order_items AS it 
        on it.order_id = ord.id
    WHERE
        ord.status <> 'cancelled'
    GROUP BY
        ord.id, ord.seller_id, ord.created_at
)

SELECT 
    sellers.name AS seller,
    montante.id AS pedido,
    montante.pedido_data,
    montante.pedido_total,
    montante.desconto_total,
    montante.percentual_desconto
FROM 
    sellers
    LEFT JOIN montante
    ON sellers.id = montante.seller_id
WHERE
    montante.percentual_desconto > 40
ORDER BY
    montante.percentual_desconto DESC
"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

# Raciocínio: Primeiro, agreguei os itens por pedido para obter o valor bruto (quantidade × preço unitário) e o desconto total, relacionando orders e order_items para analisar os valores totais e remover os pedidos cancelados. Por fim, calculei a razão entre o desconto total e o valor bruto, identificando os pedidos em que esse percentual é superior a 40%. O NULLIF evita a divisão por zero caso existam pedidos com valor bruto igual a zero.

,seller,pedido,pedido_data,pedido_total,desconto_total,percentual_desconto
0,Costa & Oliveira Atacado Distribuidora,72401,2024-01-21 02:43:43,930.30,558.02,59.98
1,Costa & Costa Atacado Distribuidora,42513,2024-03-27 02:30:57,4082.60,2447.96,59.96
2,Costa & Costa Atacado Distribuidora,36119,2024-11-14 01:05:01,21030.24,12598.00,59.90
3,Costa & Costa Atacado Distribuidora,53223,2024-05-06 01:56:47,7970.40,4773.99,59.90
4,Santos & Almeida Suprimentos Distribuidora,71165,2024-11-09 02:32:22,13385.60,8017.88,59.90
...,...,...,...,...,...,...
895,Costa & Costa Atacado Distribuidora,28496,2024-08-06 12:37:03,5935.16,2376.94,40.05
896,Costa & Costa Atacado Distribuidora,38595,2023-08-21 04:52:02,27785.26,11124.91,40.04
897,Oliveira & Santos Distribuidora Distribuidora,64306,2024-10-12 21:22:52,6907.95,2765.81,40.04
898,Souza & Ferreira Alimentos Distribuidora,32057,2024-03-10 03:51:38,24658.56,9868.74,40.02


#### Desafio 4 (Anomalia de produto)

In [0]:
# Desafio 4
# Produto com alto volume de vendas (> 1.000 unidades) que NUNCA foi o item de maior valor unitário (unit_price) dentro de nenhum pedido em que apareceu.
# Estratégia:
#   1) produto_rank: para cada item de pedido, uso RANK() particionado por order_id e ordenado por unit_price DESC -> valor_unitario_rank = 1 identifica o(s) item(ns) de maior valor unitário do pedido (RANK, e não ROW_NUMBER, para tratar empates de preço corretamente: se dois itens empatam no topo, ambos contam como 'item de maior valor').
#   2) produto_venda: volume total vendido (soma de qty) por produto, em TODOS os pedidos (a pergunta não restringe por status, diferente do Desafio 3).
#   3) top_produtos: conjunto de produtos que já foram valor_unitario_rank = 1 em pelo menos um pedido.
#   4) Seleciono produtos com total_unidades_vendidas > 1000 E que NÃO estão em top_produtos (NOT IN / NOT EXISTS).
query = """
    WITH produto_rank AS (
        SELECT
            item.order_id,
            item.product_id,
            item.unit_price,
            RANK() OVER (PARTITION BY item.order_id ORDER BY unit_price  DESC) AS valor_unitario_rank
        FROM 
            order_items AS item
    ),
    produto_venda AS (
        SELECT 
            product_id, 
            SUM(qty) AS total_unidades_vendidas, 
            COUNT(DISTINCT order_id) AS total_pedidos
        FROM 
            order_items AS item
            LEFT JOIN orders AS pedido
            ON pedido.id = item.order_id
        GROUP BY 
            product_id
    ),
    top_produtos AS (
        SELECT DISTINCT 
            product_id
        FROM 
            produto_rank
        WHERE 
            valor_unitario_rank = 1
    )

    SELECT 
        p.id AS product_id, 
        p.name, 
        p.category, 
        pv.total_unidades_vendidas, 
        pv.total_pedidos
    FROM 
        products p
        JOIN produto_venda pv ON pv.product_id = p.id
    WHERE 
        pv.total_unidades_vendidas > 1000 
        AND p.id NOT IN (SELECT product_id FROM top_produtos)
    ORDER BY 
        pv.total_unidades_vendidas DESC;
"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

# Resultado e análise de viabilidade:

# 1º Critério: A amostragem de dados tem 800 produtos, e todos já vendem, no mínimo, ~5.300 unidades. O filtro > 1.000 acaba aprovando todos os registros nesse critério.
# 2º Critério: Todos os produtos vendidos já foram, uma ou mais vezes, o produto de maior valor unitário de algum pedido, o que retiraria todos os produtos da amostra.
# Testei também a interpretação alternativa de "maior valor" = ((qty * unit_price) - discount) em vez de unit_price: também retorna vazio.

# Ou seja, a distribuição de vendas nesta base é uniforme demais para reproduzir a anomalia relatada, o que sugere dados sintéticos, e não necessariamente que a suspeita de fraude esteja errada. Antes de reportar "não existe esse produto", eu confirmaria com o time se "maior valor" pode significar margem em vez de preço, se o período/status do pedido importa e se vale flexibilizar o critério (ex.: produto que raramente, e não nunca, é o item de maior valor) para encontrar candidatos próximos.

# Questionamentos que eu levaria de volta ao time de fraude/negócio:
# A anomalia é sobre outra métrica? Talvez "maior valor" não seja unit_price bruto, mas sim margem (unit_price - unit_cost), receita líquida do item ou participação no GMV do pedido. Vale confirmar a definição exata antes de descartar a hipótese.
# O relato veio de outro recorte de tempo/status? Vale confirmar se a suspeita se baseia em um período específico (ex.: somente o último trimestre) ou em pedidos com um status específico, o que não foi aplicado por não ter sido passado como regra.
# Vale a pena flexibilizar o critério? Por exemplo, considerar um produto que raramente é o item de maior valor, digamos, em < 5% das aparições, em vez de nunca, para encontrar candidatos próximos ao perfil descrito e investigar caso a caso.
# Conclusão:

# A query está correta e resolve exatamente o que foi pedido; o resultado vazio é o resultado correto para os dados fornecidos. O ponto de atenção aqui não é técnico, mas está relacionado à regra de negócio utilizada para determinar anomalias. Antes de reportar "não encontramos esse produto", eu validaria a definição da anomalia e a representatividade da base com quem levantou a suspeita.

,product_id,name,category,total_unidades_vendidas,total_pedidos
